In [8]:
import os
import numpy as np
import h5py
from scipy.signal import resample
from sklearn.utils import shuffle
import matplotlib.pyplot as plt
import json
import warnings
from collections import defaultdict
warnings.filterwarnings("ignore")

In [2]:
import mne
from mne.preprocessing import ICA
from mne_icalabel import label_components

In [3]:
# multi-scale sampling rates
SAMPLE_RATE_LIST = [200, 100, 50]  # Hz

# fixed segment length in samples (timestamps)
SEG_LEN = 400  # number of time steps per segment
# overlap in samples (timestamps), not in seconds
OVERLAP = 200    # e.g. 200 means 50% overlap for SEG_LEN=400

assert 0 <= OVERLAP < SEG_LEN, "OVERLAP must be in [0, SEG_LEN)."

sub_folder_path = f"L{SEG_LEN}"
sub_folder_path

'L400'

In [4]:
# root dir
root = 'P-ADIC/'

In [5]:
sub_id = 1
for file_name in os.listdir(root):
    if file_name.endswith(".mat"):
        file_path = os.path.join(root, file_name)
        print(f"Read file: {file_name}")
        per_class_sub_num = 0
        # open v7.3 MATLAB file
        with h5py.File(file_path, "r") as f:
            print("Keys in the file:", list(f.keys()))
            class_group = f["#refs#"]
            # Iterate through the groups and datasets
            class_group_keys = list(class_group.keys())
            # print("Keys in this group:", class_group_keys)
            for key in class_group_keys:
                data = class_group[key]
                # check the shape of last dimension to find the data (19 channels)
                if data.shape[-1] == 19:
                    print("Subject ID:", sub_id)
                    sub_id += 1
                    per_class_sub_num += 1
                    print("---------------------")
        print(f"Subject number in this class({file_name}) is {per_class_sub_num}")
        print("------------------------------------------------\n")

Read file: alz_c1_new.mat
Keys in the file: ['#refs#', 'alz_r']
Subject ID: 1
---------------------
Subject ID: 2
---------------------
Subject ID: 3
---------------------
Subject ID: 4
---------------------
Subject ID: 5
---------------------
Subject ID: 6
---------------------
Subject ID: 7
---------------------
Subject ID: 8
---------------------
Subject ID: 9
---------------------
Subject ID: 10
---------------------
Subject ID: 11
---------------------
Subject ID: 12
---------------------
Subject ID: 13
---------------------
Subject ID: 14
---------------------
Subject ID: 15
---------------------
Subject ID: 16
---------------------
Subject ID: 17
---------------------
Subject ID: 18
---------------------
Subject ID: 19
---------------------
Subject ID: 20
---------------------
Subject ID: 21
---------------------
Subject ID: 22
---------------------
Subject ID: 23
---------------------
Subject ID: 24
---------------------
Subject ID: 25
---------------------
Subject ID: 26
-----

In [6]:
def auto_artifact_removal_iclabel_to_numpy(
    eeg_data: np.ndarray,
    sfreq: float,
    ch_names: list,
    verbose=True
):
    """
    Clean EEG data using bandpass filtering, percentile-based bad channel detection,
    ICA + ICLabel artifact removal, resampling, re-referencing, epoching, and z-score normalization.

    Args:
        eeg_data (np.ndarray): EEG data, shape (T, C).
        sfreq (float): Original sampling frequency.
        ch_names (list): List of channel names.
        verbose (bool): Verbose output.

    Returns:
        np.ndarray: Cleaned, normalized EEG data, shape (n_epochs, time_steps, channels).
    """
    # 1. Construct MNE Raw object
    info = mne.create_info(ch_names=ch_names, sfreq=sfreq, ch_types=['eeg'] * len(ch_names))
    raw = mne.io.RawArray(eeg_data.T, info)

    # 2. Set Montage
    raw.set_montage(mne.channels.make_standard_montage('standard_1020'))
    if verbose:
        print("✔ Montage set: 'standard_1020'.")
        
    # Notch filter to remove power line noise
    raw.notch_filter(freqs=50.0, verbose=False)
    if verbose:
        print("✔ Notch filter applied at 50 Hz.")

    # 3. Bandpass Filter (0.5–45 Hz)
    raw.filter(l_freq=0.5, h_freq=45.0, verbose=False)
    if verbose:
        print("✔ Bandpass filter applied (0.5–45 Hz).")

    # 4. Set average reference for ICA
    raw.set_eeg_reference('average', projection=False)
    if verbose:
        print("✔ EEG re-referenced (average) before ICA.")

    # 5. ICA + ICLabel
    raw_ica = raw.copy()
    ica = ICA(n_components=0.99, random_state=97, max_iter='auto')
    ica.fit(raw_ica)
    if verbose:
        print("✔ ICA fitted.")

    try:
        ic_labels = label_components(raw_ica, ica, method='iclabel')
        labels = ic_labels['labels']
        probs = ic_labels['y_pred_proba']

        artifact_thresholds = {
            'eye blink': 0.7,
            'muscle artifact': 0.6,
            'heart beat': 0.5,
            'line noise': 0.8,
            'channel noise': 0.9
        }

        to_exclude = [
            i for i, label in enumerate(labels)
            if label in artifact_thresholds and probs[i] >= artifact_thresholds[label]
        ]
        if to_exclude:
            ica.exclude = to_exclude
            ica.apply(raw)
            if verbose:
                print(f"✔ ICA applied. Excluded components: {to_exclude}")
        else:
            if verbose:
                print("No ICs exceeded artifact thresholds. No components excluded.")

    except Exception as e:
        if verbose:
            print(f"⚠ ICLabel failed: {e}. Proceeding without ICA-based removal.")

    return raw.get_data().T

In [7]:
def resample_time_series(data, original_fs, target_fs):
    """
    Resample each channel independently.
    data: (T_raw, C)
    return: (T_new, C)
    """
    T, C = data.shape
    new_length = int(T * target_fs / original_fs)
    resampled_data = np.zeros((new_length, C), dtype=np.float32)
    for i in range(C):
        # resample one channel
        resampled_data[:, i] = resample(data[:, i], new_length)
    return resampled_data


def compute_step(seg_len, overlap):
    """
    Compute sliding step (in samples) given segment length and overlap.
    """
    step = seg_len - overlap
    if step <= 0:
        raise ValueError(f"Invalid overlap={overlap}: step <= 0.")
    return step


def compute_num_segments(num_samples, seg_len, step):
    """
    Compute how many segments can be extracted from a sequence
    with given segment length and step.
    """
    if num_samples < seg_len:
        return 0
    return 1 + (num_samples - seg_len) // step

## PASS 1: Find total number of segments across all subjects

In [9]:
# Loop through each subject and session to preprocess the EEG data
subject_segment_counts = defaultdict(lambda: defaultdict(int))
N_total = 0
# 19 standard channels should be: 'Fp1', 'Fp2', 'F7', 'F3', 'Fz', 'F4', 'F8', 'T3', 'C3', 'Cz', 'C4', 'T4', 'T5', 'P3', 'Pz', 'P4', 'T6', 'O1', 'O2'
# For here, T7, T8 is the same to T3, T4; P7, P8 is the same to T5, T6
# So we use T7, T8, P7, P8 to replace T3, T4, T5, T6
standard_channels = ['Fp1', 'Fp2', 'F7', 'F3', 'Fz', 'F4', 'F8', 'T7', 'C3', 
                     'Cz', 'C4', 'T8', 'P7', 'P3', 'Pz', 'P4', 'P8', 'O1', 'O2']
n_channels = len(standard_channels)

# step (timestamps)
STEP = compute_step(SEG_LEN, OVERLAP)
print("SEG_LEN =", SEG_LEN, "OVERLAP =", OVERLAP, "STEP =", STEP, "\n")

RAW_SAMPLING_RATE = 500  # Original sampling rate, according to the readme file
sub_id = 1
feature_list = []
for file_name in os.listdir(root):
    if file_name.endswith(".mat"):
        file_path = os.path.join(root, file_name)
        print(f"Read file: {file_name}")
        
        # open v7.3 MATLAB file
        with h5py.File(file_path, "r") as f:
            print("Keys in the file:", list(f.keys()))
            class_group = f["#refs#"]
            # Iterate through the groups and datasets
            class_group_keys = list(class_group.keys())
            # print("Keys in this group:", class_group_keys)
            for key in class_group_keys:
                data = class_group[key]
                # check the shape of last dimension to find the data (19 channels)
                if data.shape[-1] == 19:
                    print("Subject ID:", sub_id)
                    print("Data shape:", data.shape)
                    # Convert to numpy array
                    data = np.array(data)
                    T_raw = data.shape[0]
                    for fs in SAMPLE_RATE_LIST:
                        T_res = int(T_raw * fs / RAW_SAMPLING_RATE)
                        # compute number of segments
                        n_seg = compute_num_segments(T_res, SEG_LEN, STEP)
                        subject_segment_counts[sub_id][fs] += n_seg
                        N_total += n_seg
                        print(f"fs={fs}Hz: T_res={T_res}, STEP={STEP}, n_seg={n_seg}")
                    sub_id += 1
                    print("--------End of Subject-------------\n")
        print("------------------------------------------------\n")
        
print("\n===================================")
print("Total segments N_total =", N_total)
print("Channels =", n_channels)
print("===================================")

if N_total == 0:
    raise RuntimeError("No segments found. Please check SEG_LEN / OVERLAP.")

SEG_LEN = 400 OVERLAP = 200 STEP = 200 

Read file: alz_c1_new.mat
Keys in the file: ['#refs#', 'alz_r']
Subject ID: 1
Data shape: (680502, 19)
fs=200Hz: T_res=272200, STEP=200, n_seg=1360
fs=100Hz: T_res=136100, STEP=200, n_seg=679
fs=50Hz: T_res=68050, STEP=200, n_seg=339
--------End of Subject-------------

Subject ID: 2
Data shape: (506202, 19)
fs=200Hz: T_res=202480, STEP=200, n_seg=1011
fs=100Hz: T_res=101240, STEP=200, n_seg=505
fs=50Hz: T_res=50620, STEP=200, n_seg=252
--------End of Subject-------------

Subject ID: 3
Data shape: (623552, 19)
fs=200Hz: T_res=249420, STEP=200, n_seg=1246
fs=100Hz: T_res=124710, STEP=200, n_seg=622
fs=50Hz: T_res=62355, STEP=200, n_seg=310
--------End of Subject-------------

Subject ID: 4
Data shape: (648652, 19)
fs=200Hz: T_res=259460, STEP=200, n_seg=1296
fs=100Hz: T_res=129730, STEP=200, n_seg=647
fs=50Hz: T_res=64865, STEP=200, n_seg=323
--------End of Subject-------------

Subject ID: 5
Data shape: (1262502, 19)
fs=200Hz: T_res=505000, STE

In [10]:
output_root = os.path.join('Processed', sub_folder_path, 'P-ADIC')
os.makedirs(output_root, exist_ok=True)

X_path = os.path.join(output_root, 'X.dat')
y_path = os.path.join(output_root, 'y.dat')
meta_path = os.path.join(output_root, 'meta.json')

print("X path:", X_path)
print("y path:", y_path)

# create memmap storage
X_mm = np.memmap(X_path, dtype='float32', mode='w+', shape=(N_total, SEG_LEN, n_channels))
y_mm = np.memmap(y_path, dtype='float32', mode='w+', shape=(N_total, 3))

X path: Processed\L400\P-ADIC\X.dat
y path: Processed\L400\P-ADIC\y.dat


## PASS 2: Process and store segments into memmap

In [11]:
cur = 0  # current index in memmap
total_seconds_all = 0  # used for total hours calculation

label_map = {'controls':0, 'alz':1, 'mci':2, 'dep':3, 'schiz':4}
RAW_SAMPLING_RATE = 500  # Original sampling rate, according to the readme file
sub_id = 1
for file_name in os.listdir(root):
    if file_name.endswith(".mat"):
        file_path = os.path.join(root, file_name)
        print(f"Read file: {file_name}")
        class_label = -1
        for disease_class in label_map.keys():
            if disease_class in file_name:
                print(f"Class: {disease_class}, Label: {label_map[disease_class]}")
                class_label = label_map[disease_class]
                
        # open v7.3 MATLAB file
        with h5py.File(file_path, "r") as f:
            print("Keys in the file:", list(f.keys()))
            class_group = f["#refs#"]
            # Iterate through the groups and datasets
            class_group_keys = list(class_group.keys())
            # print("Keys in this group:", class_group_keys)
            for key in class_group_keys:
                data = class_group[key]
                # check the shape of last dimension to find the data (19 channels)
                if data.shape[-1] == 19:
                    print("Subject ID:", sub_id)
                    print("Data shape:", data.shape)
                    # Convert to numpy array
                    data = np.array(data)
                    # remove the artifacts
                    ch_names = ['Fp1', 'Fp2', 'F7', 'F3', 'Fz', 'F4', 'F8', 'T3', 'C3', 'Cz', 'C4', 'T4', 'T5', 'P3', 'Pz', 'P4', 'T6', 'O1', 'O2']
                    data = auto_artifact_removal_iclabel_to_numpy(data, RAW_SAMPLING_RATE, ch_names)
                    T_raw = data.shape[0]
                    total_seconds_all += data.shape[0] / RAW_SAMPLING_RATE
                    for fs in SAMPLE_RATE_LIST:
                        # resample to target fs
                        data_resampled = resample_time_series(data, RAW_SAMPLING_RATE, fs)
                        T_res, _ = data_resampled.shape
                        print(f"fs={fs}Hz: resampled shape={data_resampled.shape}")
            
                        # overlapped sliding window with fixed STEP (in timestamps)
                        starts = np.arange(0, T_res - SEG_LEN + 1, STEP, dtype=int)
                        print(f"fs={fs}Hz: segments={len(starts)}")
            
                        for s in starts:
                            if cur >= N_total:
                                raise RuntimeError("Exceeded predicted N_total.")
            
                            X_mm[cur] = data_resampled[s:s + SEG_LEN]   # (SEG_LEN, C)
                            y_mm[cur, 0] = float(class_label)       # label
                            y_mm[cur, 1] = float(sub_id)      # Global trial ID
                            y_mm[cur, 2] = float(fs)          # sample rate (scale)
                            cur += 1
                    sub_id += 1
                    print("--------End of Subject-------------\n")
        print("------------------------------------------------\n")
        
total_hours_all = total_seconds_all / 3600.0
print("DONE: cur =", cur, " expected N_total =", N_total)
print(f"Total hours across all subjects: {total_hours_all:.2f} hours")

# flush
del X_mm
del y_mm

# save meta
meta = {
    "N": int(N_total),
    "T": SEG_LEN,
    "C": int(n_channels),
    "SAMPLE_RATE_LIST": SAMPLE_RATE_LIST,
    "OVERLAP": int(OVERLAP),   # in samples
    "STEP": int(STEP),
    "X_path": X_path,
    "y_path": y_path,
}
with open(meta_path, "w") as f:
    json.dump(meta, f, indent=2)

print("Saved meta:", meta_path)

Read file: alz_c1_new.mat
Class: alz, Label: 1
Keys in the file: ['#refs#', 'alz_r']
Subject ID: 1
Data shape: (680502, 19)
Creating RawArray with float64 data, n_channels=19, n_times=680502
    Range : 0 ... 680501 =      0.000 ...  1361.002 secs
Ready.
✔ Montage set: 'standard_1020'.
✔ Notch filter applied at 50 Hz.
✔ Bandpass filter applied (0.5–45 Hz).
EEG channel type selected for re-referencing
Applying average reference.
Applying a custom ('EEG',) reference.
✔ EEG re-referenced (average) before ICA.
Fitting ICA to data using 19 channels (please be patient, this may take a while)
Selecting by explained variance: 15 components
Fitting ICA took 6.8s.
✔ ICA fitted.
No ICs exceeded artifact thresholds. No components excluded.
fs=200Hz: resampled shape=(272200, 19)
fs=200Hz: segments=1360
fs=100Hz: resampled shape=(136100, 19)
fs=100Hz: segments=679
fs=50Hz: resampled shape=(68050, 19)
fs=50Hz: segments=339
--------End of Subject-------------

Subject ID: 2
Data shape: (506202, 19)
Cr

## Load and check the processed data

In [12]:
import json
import numpy as np

# load meta information
meta_path = meta_path  # if you already have meta_path in current notebook
with open(meta_path, "r") as f:
    meta = json.load(f)

N = meta["N"]
T = meta["T"]          # SEG_LEN
C = meta["C"]
X_path = meta["X_path"]
y_path = meta["y_path"]

print("Meta:")
print("  N =", N)
print("  T =", T)
print("  C =", C)
print("  X_path =", X_path)
print("  y_path =", y_path)
print("-------------------------------------")

# open memmap
X_mm = np.memmap(X_path, dtype='float32', mode='r', shape=(N, T, C))
y_mm = np.memmap(y_path, dtype='float32', mode='r', shape=(N, 3))

# subject_id is stored in y[:, 1]
subject_ids = np.unique(y_mm[:, 1]).astype(int)

for sid in sorted(subject_ids):
    # find all indices for this subject
    idx = np.where(y_mm[:, 1] == sid)[0]

    # compute shapes logically (do not load all X into memory)
    n_seg = len(idx)
    x_shape = (n_seg, T, C)    # logical X shape for this subject
    y_shape = (n_seg, 3)       # logical y shape for this subject

    # get sampling rates for all segments of this subject
    fs_subject = y_mm[idx, 2]  # shape (n_seg,)


    print(f"Subject ID {sid:03d}: X shape={x_shape}, y shape={y_shape}")

# option: delete memmap views
del X_mm, y_mm

Meta:
  N = 473970
  T = 400
  C = 19
  X_path = Processed\L400\P-ADIC\X.dat
  y_path = Processed\L400\P-ADIC\y.dat
-------------------------------------
Subject ID 001: X shape=(2378, 400, 19), y shape=(2378, 3)
Subject ID 002: X shape=(1768, 400, 19), y shape=(1768, 3)
Subject ID 003: X shape=(2178, 400, 19), y shape=(2178, 3)
Subject ID 004: X shape=(2266, 400, 19), y shape=(2266, 3)
Subject ID 005: X shape=(4415, 400, 19), y shape=(4415, 3)
Subject ID 006: X shape=(2121, 400, 19), y shape=(2121, 3)
Subject ID 007: X shape=(1457, 400, 19), y shape=(1457, 3)
Subject ID 008: X shape=(908, 400, 19), y shape=(908, 3)
Subject ID 009: X shape=(2135, 400, 19), y shape=(2135, 3)
Subject ID 010: X shape=(1684, 400, 19), y shape=(1684, 3)
Subject ID 011: X shape=(2622, 400, 19), y shape=(2622, 3)
Subject ID 012: X shape=(2331, 400, 19), y shape=(2331, 3)
Subject ID 013: X shape=(2199, 400, 19), y shape=(2199, 3)
Subject ID 014: X shape=(2108, 400, 19), y shape=(2108, 3)
Subject ID 015: X shap